General Processing for maps of M16/M17 from bonn sky survey and skyview

In [2]:
import os
import numpy as np

from astropy.io import fits
from astropy import units as u
from astropy.modeling.models import Gaussian2D
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
from astropy.convolution import Gaussian2DKernel, convolve

from scipy.optimize import curve_fit

import matplotlib.pyplot as plt

In [7]:
LINUX_DIRECTORY = "/scratch/nas_comap1/jgalla/MPHYS_PROJECT"

BONNSS_MAPS_LIST = [
    "data/all_maps/M16M17_cutouts_raw/Effelsberg_11cm_bg_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Effelsberg_11cm_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Effelsberg_21cm_bg_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Effelsberg_21cm_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Parkes_6cm_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Nobeyama_3cm_M16M17_cutout_raw.fits"
]

SKYVIEW_MAPS_LIST = [
    "data/all_maps/M16M17_cutouts_raw/AKARI_1.7_THz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/AKARI_2.14_THz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/AKARI_3.3_THz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/AKARI_5_THz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/ExUVEx_83_angst_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/ExUVEx_171_angst_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/ExUVEx_405_angst_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/ExUVEx_555_angst_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Fermi1_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Fermi2_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Fermi3_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Fermi4_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Fermi5_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/INTGAL_17-35_keV_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/INTGAL_35-80_keV_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Mellinger_blue_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Mellinger_green_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Mellinger_red_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/NVSS_VLA_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/PLANCK_HFI_217_GHz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/PLANCK_HFI_353_GHz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/PLANCK_HFI_545_GHz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/PLANCK_HFI_857_GHz_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Swift_BAT_SNR_75-100_keV_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Swift_BAT_SNR_100-150_keV_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Swift_BAT_SNR_150-195_keV_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/Swift_XRT_com_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/WISE_3.4micr_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/WISE_4.6micr_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/WISE_12micr_M16M17_cutout_raw.fits",
    "data/all_maps/M16M17_cutouts_raw/WISE_22micr_M16M17_cutout_raw.fits"
]

for i, dir in enumerate(BONNSS_MAPS_LIST):
    fits_file = fits.open(f"{LINUX_DIRECTORY}/{dir}")
    print("---------------------------------------------------------------------------------------------------------")
    print(dir)
    print("---------------------------------------------------------------------------------------------------------")
    for k, v in fits_file[0].header.items():
        print(f"{k}: {v}")

for i, dir in enumerate(SKYVIEW_MAPS_LIST):
    fits_file = fits.open(f"{LINUX_DIRECTORY}/{dir}")
    print("---------------------------------------------------------------------------------------------------------")
    print(dir)
    print("---------------------------------------------------------------------------------------------------------")
    for k, v in fits_file[0].header.items():
        print(f"{k}: {v}")

---------------------------------------------------------------------------------------------------------
data/all_maps/M16M17_cutouts_raw/Effelsberg_11cm_bg_M16M17_cutout_raw.fits
---------------------------------------------------------------------------------------------------------
SIMPLE: True
BITPIX: 16
NAXIS: 3
NAXIS1: 361
NAXIS2: 241
NAXIS3: 1
BSCALE: -0.0457846452244
BZERO: 2125
BLANK: -32000
CRVAL1: 17
CRPIX1: 181
CDELT1: -0.0166666666667
CROTA1: 0
CTYPE1: GLON-TAN
IRESOL1: 0.0716666666667
RESOL1: 0.0716666666667
CRVAL2: 0
CRPIX2: 121
CDELT2: 0.0166666666667
CROTA2: 0
CTYPE2: GLAT-TAN
IRESOL2: 0.0716666666667
RESOL2: 0.0716666666667
CRVAL3: 0.111317254174
CRPIX3: 1
CDELT3: 0.0001
CTYPE3: LAMBDA
OBSEPOCH: 1983.56323
EQUINOX: 1950
OBSLAT: 50.5252686
TELESCOP: EFFELSBERG 100M
BUNIT: MILLIK TB
FREQUENZ: 2695
MATRIX11: 0.0799982114931
MATRIX12: 0.490784187446
MATRIX13: -0.867600811151
MATRIX21: -0.966289197511
MATRIX22: -0.175499846719
MATRIX23: -0.188374601723
MATRIX31: -0.244715